# 04 — Modelado: Regresión Log-Log (Python / statsmodels)
**Criterio 3.1.5.I.3 (20%):** Técnica analítica en Python, código reproducible, supuestos y resultados.
**Criterio 3.1.5.I.4 (20%):** Interpretación, incertidumbre, limitaciones y no causalidad.

**Hipótesis:** El presupuesto de inversión tiene relación positiva e inelástica con las compras.
**Modelo:** `log10(Compras) = β0 + β1·log10(Gasto) + ε` con errores robustos HC1.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson, jarque_bera
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.4f}".format)
BASE = Path("../01_datos/procesados")

COLS = ["TipoCompra","codigoOC","Anio","Mes","MontoNetoItemCLP","Region",
        "Servicio","Subtitulo","ProveedorRUT","CodBIP","NombreProyecto","Proy_Gasto_MM"]
it  = pd.read_csv(BASE/"compras_presupuesto_items.csv",sep=";",low_memory=False,usecols=COLS)
res = pd.read_csv(BASE/"compras_presupuesto_resumen.csv",sep=";",encoding="utf-8-sig")

it  = it[it["MontoNetoItemCLP"]>0].copy()
it["MM"] = it["MontoNetoItemCLP"]/1e6
print("Datos listos.")

## 4.1 Función de Modelado con Diagnósticos

In [ ]:
def modelo_log(df, x, y, nombre):
    d  = df[[x,y]].dropna()
    d  = d[(d[x]>0) & (d[y]>0)]
    X  = sm.add_constant(np.log10(d[x].values))
    Y  = np.log10(d[y].values)
    m  = sm.OLS(Y, X).fit()
    mr = m.get_robustcov_results(cov_type="HC1")
    e  = m.resid
    sw = stats.shapiro(e)
    bp = het_breuschpagan(e, X)
    jb = jarque_bera(e)
    ci = m.conf_int()[1]
    return {
        "Modelo"    : nombre,
        "n"         : int(m.nobs),
        "b1"        : round(float(m.params[1]),4),
        "b0"        : round(float(m.params[0]),4),
        "EE"        : round(float(m.bse[1]),4),
        "EE_HC1"    : round(float(mr.bse[1]),4),
        "t"         : round(float(m.tvalues[1]),3),
        "p_HC1"     : float(mr.pvalues[1]),
        "IC_inf"    : round(float(ci[0]),4),
        "IC_sup"    : round(float(ci[1]),4),
        "R2"        : round(float(m.rsquared),4),
        "Shapiro_p" : round(float(sw.pvalue),4),
        "JB_p"      : round(float(jb[1]),4),
        "BP_p"      : round(float(bp[1]),4),
        "DW"        : round(float(durbin_watson(e)),3),
    }
print("Funcion modelo_log() lista.")

## 4.2 Preparación de Datasets por Modelo

In [ ]:
# M1: Proyecto BIP
proy = (it.dropna(subset=["NombreProyecto"]).groupby(["CodBIP","Anio"],as_index=False)
         .agg(Gasto_MM=("Proy_Gasto_MM","first"), Compras_MM=("MM","sum")))

# M2 y M3: Region x Servicio
rs = res[~res.Servicio.str.startswith(("MINVU","MOP otros"))].copy()

# M4: Region
reg = rs.groupby(["Region","Anio"],as_index=False)[["RS_Gasto_MM","OC"]].sum()
reg = reg[~reg.Region.isin(["Metropolitana","Interregional"])]
reg = reg[~((reg.Region=="Nuble") & (reg.Anio==2018))]

print(f"M1 proy: {len(proy):,} obs")
print(f"M2/M3 rs: {len(rs):,} obs")
print(f"M4 reg: {len(reg):,} obs")

## 4.3 Ajuste de los 4 Modelos

In [ ]:
resultados = [
    modelo_log(proy, "Gasto_MM",    "Compras_MM",          "M1 Proyecto BIP"),
    modelo_log(rs,   "RS_Gasto_MM", "Compras_MM",          "M2 Region x Servicio (total)"),
    modelo_log(rs,   "RS_Gasto_MM", "ComprasInversion_MM",  "M3 Region x Servicio (Subt.31)"),
    modelo_log(reg,  "RS_Gasto_MM", "OC",                  "M4 Region (N OC)"),
]
df_res = pd.DataFrame(resultados).set_index("Modelo")

print("=== Resultados de regresion ===")
display(df_res[["n","b1","EE_HC1","p_HC1","IC_inf","IC_sup","R2"]]
        .rename(columns={"b1":"Pendiente B1","EE_HC1":"EE HC1","p_HC1":"p-valor HC1",
                          "IC_inf":"IC 95% inf","IC_sup":"IC 95% sup"})
        .style.format({"Pendiente B1":"{:.4f}","EE HC1":"{:.4f}",
                        "p-valor HC1":"{:.2e}","R2":"{:.4f}"})
        .set_caption("Tabla 8. Resultados de regresion log-log"))

## 4.4 Diagnóstico de Supuestos

In [ ]:
diag = df_res[["n","Shapiro_p","JB_p","BP_p","DW"]].copy()
diag.columns = ["n","Shapiro-Wilk p","Jarque-Bera p","Breusch-Pagan p","Durbin-Watson"]
display(diag.style
        .format({"Shapiro-Wilk p":"{:.4f}","Jarque-Bera p":"{:.4f}",
                 "Breusch-Pagan p":"{:.4f}","Durbin-Watson":"{:.3f}"})
        .set_caption("Tabla 9. Diagnostico de supuestos"))

print(
    "Interpretacion del diagnostico:\n"
    " - Normalidad: se rechaza -> se usan EE robustos HC1\n"
    " - Homocedasticidad: no se cumple en M1-M3 -> HC1 corrige\n"
    " - Con HC1 todos los modelos tienen p < 0.001\n"
    " - Conclusion: relacion positiva e inelastica es robusta"
)
df_res.to_csv("../05_figuras/tabla_F_modelos_regresion.csv",sep=";",encoding="utf-8-sig")

## 4.5 Gráficos de Dispersión con Recta de Regresión

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13,10))
configs = [
    (proy, "Gasto_MM",    "Compras_MM",          "M1: Proyecto BIP",               axes[0,0]),
    (rs,   "RS_Gasto_MM", "Compras_MM",           "M2: Region x Servicio (total)",  axes[0,1]),
    (rs,   "RS_Gasto_MM", "ComprasInversion_MM",  "M3: Region x Servicio Subt.31",  axes[1,0]),
    (reg,  "RS_Gasto_MM", "OC",                   "M4: Region - N OC",              axes[1,1]),
]
for i,(df,x,y,titulo,ax) in enumerate(configs):
    d = df[[x,y]].dropna()
    d = d[(d[x]>0)&(d[y]>0)]
    lx = np.log10(d[x])
    ly = np.log10(d[y])
    ax.scatter(lx, ly, alpha=0.45, color="#0F4761", s=18)
    m, b = np.polyfit(lx, ly, 1)
    xr = np.linspace(lx.min(), lx.max(), 100)
    ax.plot(xr, m*xr+b, color="#EE0000", linewidth=2,
            label=f"B1={resultados[i]['b1']:.3f}  R2={resultados[i]['R2']:.3f}")
    ax.set_title(titulo, fontsize=10, fontweight="bold")
    ax.set_xlabel(f"log10({x})"); ax.set_ylabel(f"log10({y})")
    ax.legend(fontsize=8)

plt.suptitle("Regresiones Log-Log con errores HC1", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("../05_figuras/nb04_fig1_regresiones.png", dpi=130, bbox_inches="tight")
plt.show()

## 4.6 Interpretación, Incertidumbre y No Causalidad

In [ ]:
print(    'INTERPRETACION DE RESULTADOS\n'    '==============================\n'    'MAGNITUD (Beta1):\n'    '  M1=0.527 -> +10% gasto proyecto => +5.3% compras BIP\n'    '  M2=0.459 -> compras totales a nivel region-servicio\n'    '  M3=0.662 -> mayor elasticidad compras inversion\n'    '  M4=0.566 -> relacion gasto-numero OC por region\n'    '  Todos < 1: relacion INELASTICA\n'    'INCERTIDUMBRE R2 0.18-0.28:\n'    '  72-82% variabilidad explicada por factores externos\n'    'NO CAUSALIDAD: modelos CORRELACIONALES\n'    'RECOMENDACIONES:\n'    '  1. >20% Trato Directo -> auditar adquisiciones\n'    '  2. Q4 = 36.6% gasto -> revisar planificacion\n'    '  3. Solo 7.9% monto con BIP -> ampliar trazabilidad')